In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import training_lib as tl
from helperfunctions import intern_constants as ic
from helperfunctions.pretty_print import PrettyPrint as pp
from helperfunctions import controlled_env as ce
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, MaxNLocator
from typing import Dict
import pandas as pd
import numpy as np
import torch.nn as nn
from glob import glob
from pathlib import Path
import pandas as pd
#%matplotlib widget

In [ ]:
cfg_dummy = hfn.TrainConfig(config_name="further_content_part1_1", part1=True)

In [ ]:
ae_list = tl.get_model_results(src=ic.PATH_TO_BEST_MODEL_DIR, best_n=1)

In [ ]:
ae1,_,ckpt1,_,_ = ae_list[0]

In [ ]:
try:
    df_val_eval = pd.read_csv(ic.PATH_PART1_VAL_LOSS_DIR / "df_val_eval.csv")

except FileNotFoundError:
    _, val_loader , test_loader = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir= ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_PART1_VAL_SET_INJ_DIR,
    cfg=cfg_dummy,
)
    df_val_eval = tl.eval_model(model=ae1,
                                data_loader=val_loader,
                                device=ckpt1['train_config'].device,
                                loss_fn=nn.MSELoss(reduction="none")
                                )

    df_val_eval.to_csv(ic.PATH_PART1_VAL_LOSS_DIR / "df_val_eval.csv", index=False)



In [ ]:
try:
    df_test_eval = pd.read_csv(ic.PATH_PART1_TEST_LOSS_DIR/ "df_inj_eval.csv")

except FileNotFoundError:

    df_test_eval = tl.eval_model(model=ae1,
                                data_loader=test_loader,
                                device=ckpt1['train_config'].device,
                                loss_fn=nn.MSELoss(reduction="none")
                                )

    df_test_eval.to_csv(ic.PATH_PART1_TEST_LOSS_DIR/ "df_inj_eval.csv", index=False)

In [ ]:
display(df_test_eval.head())

In [ ]:
fp_best_k = ic.PATH_PART1_K_AGG_METRICS_DIR / "best_ks_per_wt.csv"

threshold_df = pd.read_csv(fp_best_k)
display(threshold_df.head(10))
theta = threshold_df["threshold"].iloc[0]
print(theta)
# theta = float(threshold_df.query(f"{ic.WT_ID} ==7 and {ic.SIGNAL_COL} == 'Mean'")["threshold"].iloc[0])

In [ ]:
cat_win_path = ic.PATH_PART1_K_AGG_METRICS_DIR / "anom_windows.csv"

cat_windows = pd.read_csv(cat_win_path)

cat_windows = cat_windows.sort_values(by=ce.AnomOverviewKeys.TS_START)

spans_all = list(zip(pd.to_datetime(cat_windows[ce.AnomOverviewKeys.TS_START]),
                     pd.to_datetime(cat_windows[ce.AnomOverviewKeys.TS_END])))




for i in spans_all:
    print(i)



In [ ]:
pp.print_loss(df_val_eval,
              dpi=500,
              y_limits=((0,0.7),(0.7,5)),
              upper_ylim_to_max=False,
              title="Val. Set 2 raw: Mean Turbine Loss per Sample",
              wt_id= None,
              ts_range=(ic.START_ANOM, '2020-03-12 00:00:00'),
              anomaly_spans=None,
              )

### Summary of Mean, Std, Median per WT

In [ ]:
def summarize_re_means_by_wt(df_eval: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df_eval
        .groupby(ic.WT_ID, as_index=False)[ic.MEAN_LOSS_PER_SAMPLE]
        .agg(mean_re = "mean",
             std_re="std",
             median_re="median",
             count="count")
    )
    return grouped.sort_values(by=ic.WT_ID).reset_index(drop=True)

def get_re_means_per_wt(df_eval:pd.DataFrame) -> Dict[int, np.ndarray]:
    ret_dict: Dict[int, np.ndarray]= {}
    for wt_id, gr in df_eval.groupby(ic.WT_ID):
        vals = gr[ic.MEAN_LOSS_PER_SAMPLE].to_numpy(dtype=float, copy=True)
        ret_dict[int(wt_id)] = vals
        
    return ret_dict

In [ ]:
summ_per_wt = summarize_re_means_by_wt(df_val_eval)
display(summ_per_wt)

In [ ]:
print(summ_per_wt["median_re"].mean())

In [ ]:
RE_per_wt = get_re_means_per_wt(df_val_eval)

wt_list = [1,2,4,5,6,7,8,9,10,11,12,13,14,15]

# for wt in wt_list:
#     wt_val = RE_per_wt[wt]
#     fig, ax = plt.subplots()
#     bins = np.arange(0.0, 0.25 + 0.0001, 0.0001)
#     ax.hist(wt_val, bins=bins)
    
#     ax.set_title(f"Distribution of RE-Means for WT {wt}")
#     ax.set_xlabel(ic.MEAN_LOSS_PER_SAMPLE)
#     ax.set_ylabel("count")
#     ax.xaxis.set_major_locator(MultipleLocator(0.0001))
#     ax.xaxis.set_major_locator(MaxNLocator(nbins=10))
#     plt.show()


In [ ]:
bins = np.arange(0.0, 0.25 + 0.001, 0.001)
data_list = [RE_per_wt[wt] for wt in wt_list]
cmap = plt.colormaps["tab20"]

fig, ax = plt.subplots()
for i,wt in enumerate(wt_list):
    wt_val = RE_per_wt[wt]
    color = cmap(i /len(wt_list))
    ax.hist(wt_val,
            bins=bins,
            histtype="step",
            density=True,
            label=f"WT {wt}",
            color=color)

ax.set_title(f"Distribution of RE-Means")
ax.set_xlabel(ic.MEAN_LOSS_PER_SAMPLE)
ax.set_ylabel("density")
ax.set_xlim(0.0, 0.25)
#ax.set_yscale("log")
ax.xaxis.set_major_locator(MultipleLocator(0.01))
ax.xaxis.set_major_locator(MaxNLocator(nbins=10))

ax.legend(ncol=3, fontsize=7)

fig.tight_layout()
plt.show()


In [ ]:
anom_span_labels = [
    "Add. Offset 1",
    "Add. Offset 2",
    "Point Anom. 1",
    "Point Anom. 2",
    "Mult. Drift",
    "Corr. Anom. 1",
    "Corr. Anom. 2",
    "Corr. Anom. 3"
]

In [ ]:
pp.print_loss(df_val_eval,
              dpi=500,
              y_limits=((0,0.2),(0.2,0.4)),
              title=f"Val2 without Injections (Base)",
              impute_label="Data Imputation",
              y_label="MSE",
              save_filename=Path(ic.PATH_PRINTS) / "part1_mean_raw.png",
              wt_id=[1],
              anom_span_label="Anomaly Windows",
              ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
              anomaly_spans=spans_all,
              mark_threshold=theta,
              #anom_span_labels=anom_span_labels,
              figsize=(9,6),
              show_mean=False
              )


In [ ]:
print("threshold value:", theta)

In [ ]:

pp.print_loss(df_test_eval,
              dpi=500,
              y_limits=((0,0.2),(0.2,40)),
              title="Val2 with Injections",
              save_filename=Path(ic.PATH_PRINTS) / "part1_mean_injected.png",
              wt_id= [1],
            #   ts_range=("2018-08-01 00:00:00","2018-10-28 00:00:00"),
              ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
              anomaly_spans=spans_all,
              line_width=0.8,
              mark_threshold=theta,
              show_mean=False,
              anom_span_labels=anom_span_labels,
              figsize=(9,6),
              impute_label="Data Imputation",
              anom_span_label="Anomaly Windows",
              y_label="MSE",
              )

In [ ]:
ratio_mse, mu_mse, sigma_mse = ce.Report.create_mu_sigma_ratio_table(
    df_base_eval=df_val_eval,
    df_inj_eval=df_test_eval,
    signals=[ic.MEAN_LOSS_PER_SAMPLE],
    anom_spans=spans_all,
    anom_cats=anom_span_labels,
    wt_id=1,
)

In [ ]:
display(ratio_mse)
display(mu_mse)
display(sigma_mse)

ltx_r = ratio_mse.to_latex(index=False, escape=False,float_format="%.2f")

with open(ic.PATH_PRINTS /"p1_mu_sigma_ratios_MSE.tex", "w", encoding="utf-8") as f:
    f.write(ltx_r)

ltx_s = sigma_mse.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_sigmas_b_MSE.tex", "w", encoding="utf-8") as f:
    f.write(ltx_s)

ltx_m = mu_mse.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_mus_b_MSE.tex", "w", encoding="utf-8") as f:
    f.write(ltx_m)

In [ ]:
def row_to_tex(df:pd.DataFrame,path:Path=Path(ic.PATH_PRINTS),fn:str="dummy"):
    cols = df.columns.tolist()
    if len(df[cols[1:]].iloc[0]) > 8:
        for i in range(len(df)):

            line = ", ".join(map(str, df[cols[1:9]].iloc[i].to_list()))
            line2 =  ", ".join(map(str, df[cols[9:]].iloc[i].to_list()))
            line = fr"$\frac{{\hat\mu_{{anom}}}}{{\hat\mu_{{base}}}}$: " + line
            line = line + fr"\\" +fr"$\frac{{\hat\sigma_{{anom}}}}{{\hat\sigma_{{base}}}}$: " + line2
            ptf = (path/f"{fn}_{df.iloc[i][0]}_{i+1}.tex")
            ptf.write_text(line, encoding="utf-8")
            #print(str(ptf))
    else:
        for i in range(len(df)):
            line = ", ".join(map(str, df[cols[1:]].iloc[i].to_list()))
            if "mu" in cols[1]:
                line = fr"$\hat\mu_{{base}}$: " + line
            else:
                line = fr"$\hat\sigma_{{base}}$: " + line
            ptf = (path/f"{fn}_{df.iloc[i][0]}_{i+1}.tex")
            ptf.write_text(line, encoding="utf-8")

row_to_tex(ratio_mse,fn="p1_mse_ratios")
row_to_tex(mu_mse,fn="p1_mse_mus")
row_to_tex(sigma_mse,fn="p1_mse_sigmas")


In [ ]:
wt = 1
df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.MEAN_LOSS_PER_SAMPLE, 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500,
            save_filename= Path(ic.PATH_PRINTS) / "part1_delta_mean.png",
            y_limits=((0,0.01),(0.01,0.25)), 
            wt_id=[1],
            title=f"Δ MSE = |base_loss - inj_loss|" ,
            values=ic.MEAN_LOSS_PER_SAMPLE,
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="MSE",
            )

In [ ]:
sigs = ["Rotor bearing temp (°C)","Stator temperature 1 (°C)","Transformer cell temperature (°C)","Generator bearing front temperature (°C)","Generator bearing rear temperature (°C)"]

In [ ]:
descr_stats = ce.Eval_Anom.descr_stats_raw_inj(
    df_raw = df_val_eval,
    df_inj = df_test_eval,
    anom_spans=spans_all,
    anom_span_labels=anom_span_labels,
    wt_id=1,
    theta = theta,
)
# fmt = {
#     "BL Coverage": lambda x: f"{x:.2f}", 
#     "Anom. Coverage": lambda x: f"{x:.2f}", 
#     "Anom. Latency": lambda x: f"{x:.2f}", 
#     "BL Latency": lambda x: f"{x:.2f}", 
#     "Latency": lambda x: np.NaN if pd.isna(x) else f"{int(x)}", 
# }
keep = ["Anomaly Type","BL Latency","Anom. Latency","BL Coverage","Anom. Coverage","Samples"]
_sigs_col = [sigs[0],sigs[0],sigs[1],sigs[1],sigs[2],sigs[3],sigs[3],sigs[3]]
_sigs = pd.Series(_sigs_col)
descr_stats["Signal"] = _sigs
keep = ["Signal"]+keep
display(keep)
descr_stats = descr_stats[keep]
descr_stats.rename(columns={"Anomaly Type": "Anomaly Category"}, inplace=True)
display(descr_stats)

ltx = descr_stats.to_latex(index=False, float_format="%.2f", escape=False)
with open(ic.PATH_PRINTS/"descr_stats_p1.tex", "w", encoding="utf-8") as f:
    f.write(ltx)

In [ ]:
best_k_per_wt_df = pd.read_csv(ic.PATH_PART1_K_AGG_METRICS_DIR/"best_ks_per_wt.csv")
display(best_k_per_wt_df)

best_k_per_wt_df.rename(columns={"WT_ID": "WT ID",
                                 "sigma": fr"$\hat\sigma$", 
                                 "threshold": fr"$\tau$", 
                                 "mu": "$\hat\mu$", 
                                 "tp": "TP",
                                 "fp": "FP",
                                 "fn": "FN",
                                 "precision": "Precision",
                                 "recall": "Recall",
                                 "f1": "F1",
                                 "FAR_per_day": "FAR per day",
                                 "latency_mean": "Latency Mean"}, inplace=True)

display(best_k_per_wt_df)
ltx = best_k_per_wt_df.to_latex(index=False, float_format="%.3f", escape=False)
with open(ic.PATH_PRINTS/"metrics_p1.tex", "w", encoding="utf-8") as f:
    f.write(ltx)

In [ ]:

df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.RE_PREFIX+ sigs[0], 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500, 
            y_limits=((0,0.3),(0.3,1)), 
            upper_ylim_to_max=False, 
            wt_id=[1],
            title=f"Δ {sigs[0][:-4]} = |base_error - inj_error|" ,
            values=f"{ic.RE_PREFIX}"+sigs[0],
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            save_filename="part1_add_signal_diff.png",
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="Squared Error per Feature"
            )

In [ ]:
fp = Path(ic.PATH_IMPUTED).glob(f"WT_ID_{wt}_*")
wt_df_raw = pd.read_csv(next(iter(fp)))

fp2 = Path(ic.PATH_PART1_VAL_SET_INJ_DIR).glob(f"wt_{wt}_*")
wt_df_inj = pd.read_csv(next(iter(fp2)))

In [ ]:
df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.RE_PREFIX+ sigs[1], 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500, 
            y_limits=((0,0.0001),(0.0001,5)), 
            wt_id=[1],
            title=f"Δ {sigs[1][:-4]} = |base_error - inj_error|" ,
            values=f"{ic.RE_PREFIX}"+sigs[1],
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            save_filename="part1_point_signal_diff.png",
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="Squared Error per Feature"
            )

In [ ]:
df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.RE_PREFIX+ sigs[2], 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500, 
            y_limits=((0,0.15),(0.15,2.5)), 
            wt_id=[1],
            title=f"Δ {sigs[2][:-4]} = |base_error - inj_error|" ,
            values=f"{ic.RE_PREFIX}"+sigs[2],
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            save_filename="part1_mult_signal_diff.png",
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="Squared Error per Feature"
            )

In [ ]:

df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.RE_PREFIX+ sigs[3], 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500, 
            y_limits=((0,0.02),(0.02,1)), 
            wt_id=[1],
            title=f"Δ {sigs[3][:-4]} = |base_error - inj_error|" ,
            values=f"{ic.RE_PREFIX}"+sigs[3],
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            save_filename="part1_corr_signal1_diff.png",
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="Squared Error per Feature"
            )

In [ ]:

df_diff = ce.Eval_Anom.build_diff_df_for_signal(wt, 
                                                signal=ic.RE_PREFIX+ sigs[4], 
                                                wt_df_raw= df_val_eval,
                                                wt_df_inj= df_test_eval,
                                                ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'))

pp.print_loss(
            df_diff, 
            dpi=500, 
            y_limits=((0,0.05),(0.05,1)),  
            wt_id=[1],
            title=f"Δ {sigs[4][:-4]} = |base_error - inj_error|" ,
            values=f"{ic.RE_PREFIX}"+sigs[4],
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            anom_span_labels=anom_span_labels,
            figsize=(9,6),
            save_filename="part1_corr_signal2_diff.png",
            show_mean=False,
            impute_label="Data Imputation",
            anom_span_label="Anomaly Windows",
            y_label="Squared Error per Feature"
            )

In [ ]:
window_to_cat = {span: label for span, label in zip(spans_all, anom_span_labels)}
df_test_wt1 = df_test_eval[df_test_eval[ic.WT_ID]==1]
df_best = ce.Eval_Anom.best_signal_in_windows(df_re=df_test_wt1, 
                                              windows= spans_all,
                                              window_to_cat=window_to_cat)
df_best = df_best.replace(r"\bRE_", "", regex=True)
display(df_best)
fmt = {"Max RE": lambda x: f"{x:.2f}"}
ltx2 = df_best.to_latex(index=False, formatters=fmt, escape=False)
with open(ic.PATH_PRINTS/"best_signals.tex", "w", encoding="utf-8") as f:
    f.write(ltx2)

In [ ]:
sigs2 = ["RE_" + sig for sig in sigs]
print(sigs2)
df_best = ce.Eval_Anom.best_signal_in_windows(df_re=df_test_wt1, 
                                              windows= spans_all,
                                              window_to_cat=window_to_cat,
                                              use_signals=sigs2)
df_best = df_best.replace(r"\bRE_", "", regex=True)
display(df_best)

ltx3 = df_best.to_latex(index=False, formatters=fmt, escape=False)
with open(ic.PATH_PRINTS/"best_signals_filtered.tex", "w", encoding="utf-8") as f:
    f.write(ltx3)

In [ ]:
print(anom_span_labels[-3:])

In [ ]:
re1 = f"Generator bearing front temperature (°C)"
re2 = f"Generator bearing rear temperature (°C)"


wt_df_inj[ic.TS_COL] = pd.to_datetime(wt_df_inj[ic.TS_COL], errors="raise")
df_wt  = wt_df_inj[wt_df_inj[ic.WT_ID] == wt]

corr_spans = [(pd.to_datetime(s), pd.to_datetime(e)) for (s,e) in spans_all[-3:]]

rows = []
for s,e in corr_spans:
    w = df_wt[(df_wt[ic.TS_COL] >= s) & (df_wt[ic.TS_COL] <= e)]
    x = w[re1].to_numpy(dtype=float)
    y = w[re2].to_numpy(dtype=float)
    rho = np.corrcoef(x,y)[0,1]
    rows.append({"ts start": s, "ts end": e, "ρ": rho})
    
corr_table = pd.DataFrame(rows)
corr_table["ρ"] = corr_table["ρ"].astype(float).round(3)

corr_table.insert(0,"Anomaly", pd.Series(anom_span_labels[-3:], index=corr_table.index))

display(corr_table)

ltx = corr_table.to_latex(index=False, escape=False)
with open(ic.PATH_PRINTS /"p1_corr_inj_test.tex", "w", encoding="utf-8") as f:
    f.write(ltx)

In [ ]:
list = zip(anom_span_labels, spans_all)

for l, t in list:
    print(f"l={l}, t={t}")

In [ ]:
display(df_diff.head())

In [ ]:
meta_cols = [ic.TS_COL, ic.WT_ID]
re_sigs = [ic.MEAN_LOSS_PER_SAMPLE]+[f"{ic.RE_PREFIX}{sig}" for sig in sigs]
total_cols = meta_cols + re_sigs



base = df_val_eval[total_cols].sort_values(meta_cols).reset_index(drop=True)
inj = df_test_eval[total_cols].sort_values(meta_cols).reset_index(drop=True)



diff = (base[re_sigs] - inj[re_sigs]).abs()
diff_df = pd.concat([inj[meta_cols], diff], axis=1)

display(diff_df.columns)
assert base[meta_cols].equals(inj[meta_cols])

display(diff_df)
display(sigs)

display(diff_df[f"{ic.RE_PREFIX}{sigs[0]}"])

In [ ]:
ratios,mu_b, sigma_b = ce.Report.create_mu_sigma_ratio_table(
    df_base_eval=df_val_eval,
    df_inj_eval=diff_df,
    signals=[ic.MEAN_LOSS_PER_SAMPLE]+sigs,
    anom_spans=spans_all,
    anom_cats=anom_span_labels,
    wt_id=1,
)

In [ ]:

display(ratios)
display(sigma_b)
display(mu_b)

In [ ]:
ratios = ratios.rename(columns={"Signal": "Target"})
ratios["Target"] = ratios["Target"].replace({
    "MSE": "Δ MSE",
    "Rotor bearing temp (°C)": "Δ Rotor bearing temp (°C)",
    "Stator temperature 1 (°C)": "Δ Stator temperature 1 (°C)",
    "Transformer cell temperature (°C)": "Δ Transformer cell temperature (°C)",
    "Generator bearing front temperature (°C)": "Δ Generator bearing front temperature (°C)",
    "Generator bearing rear temperature (°C)": "Δ Generator bearing rear temperature (°C)",
})
df_ratios1 = ratios.iloc[:, :9].copy()
df_ratios2 = ratios.loc[:,["Target"] + ratios.columns[9:].tolist()].copy()
display(df_ratios1)
display(df_ratios2)

In [ ]:
ltx_r = df_ratios1.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_mu_sigma_ratios.tex", "w", encoding="utf-8") as f:
    f.write(ltx_r)

ltx_r2 = df_ratios2.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_mu_sigma_ratios2.tex", "w", encoding="utf-8") as f:
    f.write(ltx_r2)


ltx_s = sigma_b.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_sigmas_b.tex", "w", encoding="utf-8") as f:
    f.write(ltx_s)

ltx_m = mu_b.to_latex(index=False, escape=False,float_format="%.3f")

with open(ic.PATH_PRINTS /"p1_mus_b.tex", "w", encoding="utf-8") as f:
    f.write(ltx_m)

In [ ]:
row_to_tex(
    df=ratios,
    fn="p1_diffs_ratio",
)
row_to_tex(
    df=mu_b,
    fn="p1_diffs_mu",
)
row_to_tex(
    df=sigma_b,
    fn="p1_diffs_sigma"
)

In [ ]:
sig = "Ambient temperature (converter) (°C)"

pp.print_loss(
            df_test_eval,
            #df_val_eval, 
            dpi=500, 
            y_limits=((0,1),(1,20)), 
            wt_id=[1],
            title=f"WT {wt}: delta_{sig[:-4]} = |raw_sig - inj_sig|" ,
            values=f"{ic.RE_PREFIX}{sig}",
            ts_range=(ic.START_ANOM, '2020-02-23 00:00:00'),
            anomaly_spans=spans_all,
            y_label="Min-max scaled"
            )

In [ ]:
display(spans_all)

In [ ]:

fw= spans_all[0]
print(fw)
df_filtered = df_test_eval[df_test_eval[ic.WT_ID] == wt]
df_filtered = df_filtered[(df_filtered[ic.TS_COL]>= fw[0]) & (df_filtered[ic.TS_COL] <= fw[1])]

df_filtered = df_filtered["RE_Hub temperature (°C)"].max()
display(df_filtered)


In [ ]:
display(spans_all)

In [ ]:
meta_cols = [ic.TS_COL, ic.WT_ID]
re_sigs = [ic.MEAN_LOSS_PER_SAMPLE]+[f"{ic.RE_PREFIX}{sig}" for sig in hfn.load_feature_order()]
total_cols = meta_cols + re_sigs



base = df_val_eval[total_cols].sort_values(meta_cols).reset_index(drop=True)
inj = df_test_eval[total_cols].sort_values(meta_cols).reset_index(drop=True)



diff = (base[re_sigs] - inj[re_sigs]).abs()
diff_all_sigs = pd.concat([inj[meta_cols], diff], axis=1)

In [ ]:
pp.plot_heatmap_RE(diff_all_sigs,
                   windows=spans_all,
                   point_width=8,
                   point_signal="Stator temperature 1 (°C)",
                   cmap="magma",
                   use_lognorm=False,
                   use_manual_color_scale=True,
                   vmax=0.002,
                   title="RE Heatmap: |RE_anom - RE_base|")